# TASK -**1**

Autonomous Web Scraper & Data Extractor
Goal: Extract structured data from multi-page web portals using BeautifulSoup4.
Build a Python scraper that navigates paginated websites (e.g. e-commerce product listings or job postings), parses HTML tags, and exports structured JSON/CSV records.
Python 3requestsBeautifulSoup4
Suggested Approach: Use requests with user-agent headers, BeautifulSoup parsing, pagination loops, and retry/exception handling.
Deliverable: Clean CSV/JSON dataset containing at least 100+ extracted records with title, price/salary, and metadata.

In [1]:
import requests

# 1. Target URL we want to inspect
url = "https://quotes.toscrape.com/page/1/"

# 2. Tell the website we are visiting using a standard web browser header
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36"
}

# 3. Fetch the webpage content
response = requests.get(url, headers=headers)

# 4. Check if the website answered successfully (Status 200 means OK)
print("HTTP Status Code:", response.status_code)

# 5. Print the first 300 characters of raw HTML to see what came back
print("\nFirst 300 characters of raw HTML:")
print(response.text[:300])

HTTP Status Code: 200

First 300 characters of raw HTML:
<!DOCTYPE html>
<html lang="en">
<head>
	<meta charset="UTF-8">
	<title>Quotes to Scrape</title>
    <link rel="stylesheet" href="/static/bootstrap.min.css">
    <link rel="stylesheet" href="/static/main.css">
    
    
</head>
<body>
    <div class="container">
        <div class="row header-box">



In [2]:
from bs4 import BeautifulSoup

# 1. Feed the raw HTML (response.text) to BeautifulSoup
# "html.parser" is Python's built-in tool for reading HTML structure
soup = BeautifulSoup(response.text, "html.parser")

# 2. Extract the page title tag directly using dot notation
page_title = soup.title.get_text()
print("Page Title:", page_title)

# 3. Find the very first quote text on the page using a CSS selector
# On this site, quotes live inside <span class="text">
first_quote = soup.select_one("span.text").get_text()
print("\nFirst Quote Found:")
print(first_quote)

Page Title: Quotes to Scrape

First Quote Found:
“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”


In [3]:
# 1. Find ALL containers with class "quote"
quote_cards = soup.select("div.quote")
print(f"Total item cards found on this page: {len(quote_cards)}\n")

# 2. Create a empty list to store our clean data
all_page_quotes = []

# 3. Loop through each card one by one
for index, card in enumerate(quote_cards, start=1):
    # Extract the quote text inside <span class="text">
    quote_text = card.select_one("span.text").get_text(strip=True)

    # Extract the author name inside <small class="author">
    author_name = card.select_one("small.author").get_text(strip=True)

    # Extract all tag names inside <a class="tag">
    tag_elements = card.select("a.tag")
    tags = [tag.get_text(strip=True) for tag in tag_elements]

    # Store clean record as a dictionary
    item_data = {
        "quote": quote_text,
        "author": author_name,
        "tags": tags
    }
    all_page_quotes.append(item_data)

    # Print the first 2 items to inspect the output
    if index <= 2:
        print(f"--- Item #{index} ---")
        print("Quote: ", quote_text)
        print("Author:", author_name)
        print("Tags:  ", tags)
        print()

Total item cards found on this page: 10

--- Item #1 ---
Quote:  “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
Author: Albert Einstein
Tags:   ['change', 'deep-thoughts', 'thinking', 'world']

--- Item #2 ---
Quote:  “It is our choices, Harry, that show what we truly are, far more than our abilities.”
Author: J.K. Rowling
Tags:   ['abilities', 'choices']



In [4]:
# 1. Find the 'Next' button on the current page
# In HTML: <li class="next"><a href="/page/2/">Next</a></li>
next_button = soup.select_one("li.next a")

if next_button:
    # 2. Get the raw href value (e.g., "/page/2/")
    relative_url = next_button["href"]
    print("Relative link found:", relative_url)

    # 3. Join the relative link with base URL to make a full clickable URL
    base_url = "https://quotes.toscrape.com"
    full_next_url = requests.compat.urljoin(base_url, relative_url)
    print("Full next page URL: ", full_next_url)
else:
    print("No next button found! We reached the end.")

Relative link found: /page/2/
Full next page URL:  https://quotes.toscrape.com/page/2/


In [7]:
import csv
import json
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd
from google.colab import files

# --- Configuration ---
base_url = "https://quotes.toscrape.com"
current_url = "https://quotes.toscrape.com/page/1/"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36"
}

scraped_records = []
page_count = 1
# Set max_pages to 10 (10 pages x 10 quotes = 100 quotes)
max_pages = 10

print("🚀 Starting scraper to collect 100+ records...\n")

# --- Pagination Loop ---
while current_url and page_count <= max_pages:
    print(f"📄 Scraping Page {page_count}: {current_url}")

    # 1. Fetch HTML
    response = requests.get(current_url, headers=headers)
    if response.status_code != 200:
        print(f"❌ HTTP Error {response.status_code}. Stopping.")
        break

    # 2. Parse HTML
    soup = BeautifulSoup(response.text, "html.parser")
    quote_cards = soup.select("div.quote")

    # 3. Extract items
    for card in quote_cards:
        quote_text = card.select_one("span.text").get_text(strip=True)
        author_name = card.select_one("small.author").get_text(strip=True)
        tag_elements = card.select("a.tag")
        tags = [tag.get_text(strip=True) for tag in tag_elements]

        scraped_records.append({
            "quote": quote_text,
            "author": author_name,
            "tags": ", ".join(tags)
        })

    print(f"   Collected {len(quote_cards)} items from this page. Total so far: {len(scraped_records)}")

    # 4. Find Next Page
    next_button = soup.select_one("li.next a")
    if next_button and "href" in next_button.attrs:
        relative_url = next_button["href"]
        current_url = requests.compat.urljoin(base_url, relative_url)
        page_count += 1
        time.sleep(1)  # Polite delay
    else:
        print("No further pages available.")
        current_url = None

print(f"\n✅ Scraping finished! Total records collected: {len(scraped_records)}")

# --- Save to Files ---
with open("quotes_100_records.json", "w", encoding="utf-8") as f:
    json.dump(scraped_records, f, indent=4)

with open("quotes_100_records.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["quote", "author", "tags"])
    writer.writeheader()
    writer.writerows(scraped_records)

print("📁 Files created: 'quotes_100_records.csv' and 'quotes_100_records.json'")

🚀 Starting scraper to collect 100+ records...

📄 Scraping Page 1: https://quotes.toscrape.com/page/1/
   Collected 10 items from this page. Total so far: 10
📄 Scraping Page 2: https://quotes.toscrape.com/page/2/
   Collected 10 items from this page. Total so far: 20
📄 Scraping Page 3: https://quotes.toscrape.com/page/3/
   Collected 10 items from this page. Total so far: 30
📄 Scraping Page 4: https://quotes.toscrape.com/page/4/
   Collected 10 items from this page. Total so far: 40
📄 Scraping Page 5: https://quotes.toscrape.com/page/5/
   Collected 10 items from this page. Total so far: 50
📄 Scraping Page 6: https://quotes.toscrape.com/page/6/
   Collected 10 items from this page. Total so far: 60
📄 Scraping Page 7: https://quotes.toscrape.com/page/7/
   Collected 10 items from this page. Total so far: 70
📄 Scraping Page 8: https://quotes.toscrape.com/page/8/
   Collected 10 items from this page. Total so far: 80
📄 Scraping Page 9: https://quotes.toscrape.com/page/9/
   Collected 10 it

In [8]:
# Load CSV to verify row count
df = pd.read_csv("quotes_100_records.csv")
print(f"Total Rows in CSV: {len(df)}")
print("\nFirst 5 rows preview:")
print(df.head())

# Download CSV directly to your computer
files.download("quotes_100_records.csv")
files.download("quotes_100_records.json")

Total Rows in CSV: 100

First 5 rows preview:
                                               quote           author  \
0  “The world as we have created it is a process ...  Albert Einstein   
1  “It is our choices, Harry, that show what we t...     J.K. Rowling   
2  “There are only two ways to live your life. On...  Albert Einstein   
3  “The person, be it gentleman or lady, who has ...      Jane Austen   
4  “Imperfection is beauty, madness is genius and...   Marilyn Monroe   

                                           tags  
0        change, deep-thoughts, thinking, world  
1                            abilities, choices  
2  inspirational, life, live, miracle, miracles  
3              aliteracy, books, classic, humor  
4                    be-yourself, inspirational  


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>